<a href="https://colab.research.google.com/github/Umang-Raval/DSSE_Assignment1/blob/main/Week3/Jinaai/%5BREAD_ONLY%5DDS4SE26Week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**0. Import requireed modules**

In [65]:
import numpy as np
import torch
import pandas as pd
import re
import os
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from pathlib import Path
from google.colab import userdata

**1. Provide the path of java files and load their source code into a list**

In [66]:
SOURCE_CODE_DIR = Path("/content/resourcemanager")
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify files found
java_files = list(SOURCE_CODE_DIR.rglob("*.java"))
print(f"Found {len(java_files)} Java files")

Found 1498 Java files


In [67]:
files_list = []
file_paths_list = []

for file_path in SOURCE_CODE_DIR.rglob('*.java'):
    if file_path.is_file():
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            files_list.append(f.read())
            file_paths_list.append(file_path)

print(f"Loaded {len(files_list)} Java files")

Loaded 1498 Java files


**2. Retrieve the Hugging Face token securely from Colab's "Secrets" tab (the key icon on the left).**

In [68]:
try:
    hf_token = userdata.get('HF_TOKEN')
    print("Token loaded successfully!")
except Exception:
    print("WARNING: HF_TOKEN not found in Colab Secrets.")
    hf_token = None

Token loaded successfully!


Get full class names from Java files (needed for RSF matching)


In [69]:
def get_full_class_name(file_path):
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
        package_match = re.search(r'^\s*package\s+([^;]+);', content, re.MULTILINE)
        package = package_match.group(1).strip() if package_match else ""
        class_name = Path(file_path).stem
        return f"{package}.{class_name}" if package else class_name

full_class_names = [get_full_class_name(p) for p in file_paths_list]
print(f"Total classes: {len(full_class_names)}")
print("Sample:", full_class_names[:3])

Total classes: 1498
Sample: ['org.apache.hadoop.yarn.server.resourcemanager.RMInfoMXBean', 'org.apache.hadoop.yarn.server.resourcemanager.RMFatalEvent', 'org.apache.hadoop.yarn.server.resourcemanager.NodesListManager']


**3. Hardware optimization (Quantization)**

In [70]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

**4. Generate embeddings**

In [71]:
embedding_model_name = "jinaai/jina-code-embeddings-0.5b"

tokenizer = AutoTokenizer.from_pretrained(
    embedding_model_name, token=hf_token, trust_remote_code=True
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

try:
    model = AutoModel.from_pretrained(
        embedding_model_name,
        token=hf_token,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    print("Model loaded with 4-bit quantization.")
except Exception as e:
    print(f"Quantization failed: {e}")
    print("Falling back to standard loading...")
    model = AutoModel.from_pretrained(
        embedding_model_name, token=hf_token, trust_remote_code=True
    )
    model.to(device)

Using device: cuda
Quantization failed: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`
Falling back to standard loading...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

**5. Construct the semantic similarity matrix**

In [72]:
def embed_source_code(code_files):
    embeddings = []
    model.eval()
    for i, code in enumerate(code_files):
        inputs = tokenizer(
            code, padding=True, truncation=True,
            max_length=512, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            # .to(torch.float32) fixes BFloat16 error
            # .mean(dim=1) = mean pooling across all tokens
            code_embedding = outputs.last_hidden_state.to(torch.float32).mean(dim=1).squeeze().cpu().numpy()

        embeddings.append(code_embedding)

        if i % 50 == 0:
            print(f"  Embedded {i}/{len(code_files)} files...")

    return np.array(embeddings)

In [ ]:
# embeddings = embed_source_code(files_list)
# semantic_matrix = cosine_similarity(embeddings)
semantic_matrix = np.zeros((len(files_list), len(files_list)))

if len(files_list) > 0:
    print(f"Generating embeddings for {len(files_list)} files...")
    try:
        embeddings_array = embed_source_code(files_list)
        semantic_matrix = cosine_similarity(embeddings_array)
        print(f"Semantic matrix shape: {semantic_matrix.shape}")
    except Exception as e:
        print(f"Error: {e}")

Generating embeddings for 1498 files...
  Embedded 0/1498 files...


**6. Construct the structural similarity matrix**

In [ ]:
# scan the filtered .rsf dependency file from week 1 to construct a matrix where each entry is the number of packages each pair of files depend on.
# executing this cell produces 'struct_matrix_raw'
# Upload your filtered RSF file to /content/ first!
rsf_path = "/content/filtered_rsf.rsf"  # ← change to your actual rsf filename

num_files = len(full_class_names)
struct_matrix_raw = np.zeros((num_files, num_files))
dependencies = {}

if not Path(rsf_path).exists():
    print(f"ERROR: {rsf_path} not found! Upload your RSF file to /content/")
else:
    with open(rsf_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 3 and parts[0] == 'depends':
                source, target = parts[1], parts[2]
                if source not in dependencies:
                    dependencies[source] = set()
                dependencies[source].add(target)

    for i in range(num_files):
        for j in range(i, num_files):
            set_i = dependencies.get(full_class_names[i], set())
            set_j = dependencies.get(full_class_names[j], set())
            common = len(set_i.intersection(set_j))
            struct_matrix_raw[i, j] = struct_matrix_raw[j, i] = common

    print(f"Structural matrix shape: {struct_matrix_raw.shape}")
    print(f"Max overlap: {struct_matrix_raw.max()}")

**7. Normalize the structural matrix**

In [ ]:
max_overlap = struct_matrix_raw.max()
struct_matrix = struct_matrix_raw / max_overlap if max_overlap > 0 else struct_matrix_raw.copy()
np.fill_diagonal(struct_matrix, 1.0)
print(f"Structural matrix normalized. Max: {struct_matrix.max()}")

**8. Combine the two matrices into one similarity matrix then apply complement.**

In [ ]:
ALPHA = 0.5  # 50% structural, 50% semantic
combined_similarity = (ALPHA * struct_matrix) + ((1 - ALPHA) * semantic_matrix)
distance_matrix = 1.0 - combined_similarity
np.fill_diagonal(distance_matrix, 0)

TARGET_NUM_CLUSTERS = 10
clusterer = AgglomerativeClustering(
    n_clusters=TARGET_NUM_CLUSTERS, metric='precomputed', linkage='complete'
)
clusters = clusterer.fit_predict(distance_matrix)
print(f"Clustering done! {len(clusters)} files → {TARGET_NUM_CLUSTERS} clusters")

In [ ]:
# PCA scatter plot
pca = PCA(n_components=2)
reduced_data = pca.fit_transform(distance_matrix)

plot_df = pd.DataFrame({
    'x': reduced_data[:, 0],
    'y': reduced_data[:, 1],
    'Cluster': [f'Cluster {c}' for c in clusters]
})

sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 10))
sns.scatterplot(data=plot_df, x='x', y='y', hue='Cluster',
                palette='tab10', s=100, alpha=0.8, edgecolor='w')
plt.title('Yarn ResourceManager: Semantic + Structural Clustering (Jina)', fontsize=16)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Clusters')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "clustering_jina.png", bbox_inches='tight')
plt.show()

In [ ]:
results_df = pd.DataFrame({
    'Full_Class_Name': full_class_names[:len(clusters)],
    'Cluster_ID': clusters
})

results_df.to_csv(OUTPUT_DIR / "clustering_results_jina.csv", index=False)

rsf_output = OUTPUT_DIR / "final_clustering_jina.rsf"
with open(rsf_output, 'w') as f:
    for _, row in results_df.iterrows():
        f.write(f"contain Cluster_{row['Cluster_ID']} {row['Full_Class_Name']}\n")

print(f"Saved RSF: {rsf_output}")
display(results_df.head(10))

In [ ]:
MODELS = ["jinaai/jina-code-embeddings-0.5b", "nomic-ai/CodeRankEmbed"]
ALPHAS = [0.3, 0.5, 0.7]
CLUSTERS_LIST = [8, 12, 16]
BASE_OUTPUT = Path("/content/experiment_results")

def run_full_pipeline():
    if BASE_OUTPUT.exists():
        shutil.rmtree(BASE_OUTPUT)

    for model_name in MODELS:
        print(f"\n>>> Model: {model_name}")
        curr_tokenizer = AutoTokenizer.from_pretrained(
            model_name, token=hf_token, trust_remote_code=True
        )
        curr_model = AutoModel.from_pretrained(
            model_name, token=hf_token, trust_remote_code=True
        ).to(device)
        curr_model.eval()

        # Generate embeddings
        embs = []
        for i, code in enumerate(files_list):
            inp = curr_tokenizer(code, padding=True, truncation=True,
                                 max_length=512, return_tensors="pt").to(device)
            with torch.no_grad():
                out = curr_model(**inp)
                emb = out.last_hidden_state.to(torch.float32).mean(dim=1).squeeze().cpu().numpy()
            embs.append(emb)
            if i % 50 == 0:
                print(f"  {i}/{len(files_list)} embedded...")

        curr_semantic = cosine_similarity(np.array(embs))

        # Try all alpha + cluster combinations
        for alpha in ALPHAS:
            for n_clusters in CLUSTERS_LIST:
                run_dir = BASE_OUTPUT / model_name.split('/')[-1] / f"alpha_{alpha}_clusters_{n_clusters}"
                run_dir.mkdir(parents=True, exist_ok=True)

                sim = (alpha * struct_matrix) + ((1 - alpha) * curr_semantic)
                dist = 1.0 - sim
                np.fill_diagonal(dist, 0)

                labels = AgglomerativeClustering(
                    n_clusters=n_clusters, metric='precomputed', linkage='complete'
                ).fit_predict(dist)

                # Save CSV
                df = pd.DataFrame({'Class': full_class_names, 'Cluster': labels})
                df.to_csv(run_dir / "results.csv", index=False)

                # Save RSF
                with open(run_dir / "output.rsf", 'w') as f:
                    for _, row in df.iterrows():
                        f.write(f"contain Cluster_{row['Cluster']} {row['Class']}\n")

                # Save plot
                coords = PCA(n_components=2).fit_transform(dist)
                viz_df = pd.DataFrame({
                    'PCA 1': coords[:, 0], 'PCA 2': coords[:, 1],
                    'Cluster': [f"C{l}" for l in labels]
                })
                plt.figure(figsize=(12, 8))
                sns.scatterplot(data=viz_df, x='PCA 1', y='PCA 2',
                                hue='Cluster', palette='tab20', alpha=0.8, edgecolors='k')
                plt.title(f"{model_name.split('/')[-1]} | Alpha: {alpha} | Clusters: {n_clusters}")
                plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                plt.savefig(run_dir / "visualization.png", bbox_inches='tight')
                plt.close()

                print(f"  Saved: alpha={alpha}, clusters={n_clusters}")

run_full_pipeline()
print(f"\nAll done! Results in: {BASE_OUTPUT}")

In [ ]:
from google.colab import files

shutil.make_archive("clustering_experiment_results", 'zip', BASE_OUTPUT)
files.download("clustering_experiment_results.zip")
print("Downloading results zip...")

**9. Apply clustering**

In [ ]:
TARGET_NUM_CLUSTERS = [HYPER_PARAMETER] # how such parameter could be optimized?
clusterer = AgglomerativeClustering(n_clusters=TARGET_NUM_CLUSTERS, metric='precomputed', linkage='complete') # what is 'precomputed' & 'linkage'?
clusters = clusterer.fit_predict(distance_matrix)

**10. Visulizations**

In [ ]:
# According to the applied algorithm documentation, provide any visualizations for better understanding of how the clusters are formed.